In [2]:
import os
import cv2
import json
import glob
import numpy as np
from tqdm import tqdm

# Parse video info

In [3]:
videos_dir = 'E:/AIChallenge24_Dataset3/AIC_Video'
all_video_paths = dict()
for part in sorted(os.listdir(videos_dir)):
    data_part = part.split('_')[-1] # L01, L02 for ex
    all_video_paths[data_part] =  dict()

for data_part in sorted(all_video_paths.keys()):
    data_part_path = f'{videos_dir}/Videos_{data_part}/video'
    video_paths = sorted(os.listdir(data_part_path))
    video_ids = [video_path.replace('.mp4', '').split('_')[-1] for video_path in video_paths]
    for video_id, video_path in zip(video_ids, video_paths):
        video_path_full = f'{data_part_path}/{video_path}'
        all_video_paths[data_part][video_id] = video_path_full

# Sampling Utils

In [4]:
def sample_frames_from_shot(start_idx, end_idx):
    '''
    intervals = np.linspace(start=start_idx, stop=end_idx, num=n_frames+1).astype(int)
    ranges = []
    for idx, interv in enumerate(intervals[:-1]):
        ranges.append((interv, intervals[idx + 1]))
    frame_idxs = [(x[0] + x[1]) // 2 for x in ranges]
    '''
    idx_first = start_idx
    idx_end = end_idx
    idx_03 = start_idx + int((end_idx-start_idx)/3)
    idx_06 = start_idx + int(2*(end_idx-start_idx)/3)
    frame_idxs = [idx_first, idx_03, idx_06, idx_end]
    return frame_idxs

# CutFrame

In [6]:
scene_json_dirs = 'E:/AIChallenge24_Dataset3/Data_Extract_FULL/SceneJSON'
save_dir_all = 'E:/AIChallenge24_Dataset3/Keyframes_Extra/Keyframes'
if not os.path.exists(save_dir_all):
    os.mkdir(save_dir_all)

for key in all_video_paths.keys():
    if key == 'L30':
        save_dir = f'{save_dir_all}/{key}_extra'

        if not os.path.exists(save_dir):
            os.mkdir(save_dir)
        
        video_paths_dict = all_video_paths[key]
        video_ids = sorted(video_paths_dict.keys())
        for video_id in tqdm(video_ids):
            if video_id >= 'V001':
                video_path = video_paths_dict[video_id]
                video_scene_path =  f'{scene_json_dirs}/{key}/{video_id}.json'
                
                with open(video_scene_path, 'r') as f:
                    video_scenes = json.load(f)
                
                if not os.path.exists(f'{save_dir}/{video_id}'):
                    os.mkdir(f'{save_dir}/{video_id}')
                
                cap = cv2.VideoCapture(video_path)
                for i, shot in enumerate(tqdm(video_scenes)):
                    shot_frames_id = sample_frames_from_shot(shot[0], shot[1])
                    for index in shot_frames_id:
                        cap.set(cv2.CAP_PROP_POS_FRAMES, index)
                        filename = "{}/{:0>6d}.jpg".format(f'{save_dir}/{video_id}', index)
                        ret, frame = cap.read()
                        if ret:
                            if not cv2.imwrite(filename, frame):
                                print('fail save')
                        else:
                            pass
                cap.release()

100%|██████████| 96/96 [14:53<00:00,  9.31s/it]


In [ ]:
# => Thực hiện khi hoàn thiện hết việc xuất dữ liệu ảnh ra và đã nén + up lên GGDrive hoặc Kaggle
# => Thao tác này giảm chất lượng ảnh để phù hợp chạy FE (để trong folder frontend/ai/public/data)
from PIL import Image
import os

# Đường dẫn đến thư mục chứa hình ảnh
input_folder = r'F:\DataExtract_FULL\Keyframes\L01_extra\V012'
output_folder = r'F:\DataExtract_FULL\Keyframes\L01_extra\V012'

# Tạo thư mục đầu ra nếu chưa tồn tại
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Giảm dung lượng các hình ảnh
for filename in os.listdir(input_folder):
    if filename.endswith(('.jpg', '.jpeg', '.png')):
        img = Image.open(os.path.join(input_folder, filename))
        img = img.convert("RGB")  # Đảm bảo ảnh là RGB
        output_path = os.path.join(output_folder, filename)
        
        # Lưu ảnh với chất lượng nén giảm
        img.save(output_path, optimize=True, quality=50)  # Quality từ 0-100
